# NarratoAI Video Understanding

Enable GPU for this Kaggle Notebook. Attach the NarratoAI video-understanding Dataset, then run all cells. The notebook writes `event_timeline.json`, `candidate_clips.json`, `quality_report.json`, and optional `rough_cut.mp4` to `/kaggle/working/narratoai_outputs/`.

In [ ]:
!pip install -q -U "transformers>=4.49,<4.50" "accelerate>=0.34,<1" "pillow>=10,<12" "opencv-python-headless>=4.9,<5"


In [ ]:
import json


In [ ]:
from pathlib import Path


In [ ]:
input_root = Path('/kaggle/input')


In [ ]:
input_dirs = [d for d in input_root.iterdir() if d.is_dir()]


In [ ]:
if not input_dirs:
    raise FileNotFoundError('No dataset found at /kaggle/input - attach the NarratoAI Kaggle Dataset first.')


In [ ]:
configs = []
for input_dir in input_dirs:
    configs.extend(sorted(input_dir.glob('task_config.json')))
    configs.extend(sorted(input_dir.glob('*/task_config.json')))
    configs.extend(sorted(input_dir.glob('*/*/task_config.json')))


In [ ]:
if not configs:
    seen = ', '.join(path.name for path in input_dirs)
    raise FileNotFoundError(f'No task_config.json found under /kaggle/input. Seen input dirs: {seen}')


In [ ]:
config_path = configs[-1]


In [ ]:
dataset_dir = next((parent for parent in config_path.parents if parent.parent == input_root), config_path.parent)


In [ ]:
runner_candidates = [
    Path('/kaggle/working/narratoai_kaggle_video_runner.py'),
    config_path.parent / 'narratoai_kaggle_video_runner.py',
    dataset_dir / 'narratoai_kaggle_video_runner.py',
]
runner_path = next((path for path in runner_candidates if path.exists()), runner_candidates[0])


In [ ]:
print('Dataset:', dataset_dir.name)
print('Config:', config_path)
print('Runner:', runner_path)


In [ ]:
if not runner_path.exists():
    raise FileNotFoundError(f'Runner not found: {runner_path}')


In [ ]:
config = json.loads(config_path.read_text(encoding='utf-8'))


In [ ]:
def resolve_dataset_file(value):
    value = str(value or '').strip()
    if not value:
        return ''
    path = Path(value)
    candidates = []
    if path.is_absolute():
        candidates.append(path)
    candidates.extend([
        config_path.parent / path,
        dataset_dir / path,
        dataset_dir / 'input' / path.name,
    ])
    for candidate in candidates:
        if candidate.exists():
            return str(candidate.resolve())
    matches = sorted(input_root.rglob(path.name))
    if matches:
        return str(matches[0].resolve())
    return value


In [ ]:
config['video_file'] = resolve_dataset_file(config.get('video_file'))
config['subtitle_file'] = resolve_dataset_file(config.get('subtitle_file'))


In [ ]:
runtime_config_path = Path('/kaggle/working/task_config_runtime.json')
runtime_config_path.write_text(json.dumps(config, ensure_ascii=False, indent=2), encoding='utf-8')
config_path = runtime_config_path


In [ ]:
print('Resolved video:', config.get('video_file'))
print('Resolved subtitle:', config.get('subtitle_file'))


In [ ]:
import os
import subprocess


In [ ]:
os.environ['NARRATOAI_KAGGLE_OUTPUT_DIR'] = '/kaggle/working/narratoai_outputs'


In [ ]:
subprocess.run(['python', str(runner_path), '--config', str(config_path)], check=True)


In [ ]:
from pathlib import Path
output_dir = Path('/kaggle/working/narratoai_outputs')
print('Outputs:')
for path in sorted(output_dir.glob('*')):
    print(path)
